In [1]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
from torch import Tensor
from torch.nn.modules import Module
import torch.optim as optim

import torchgeo
from torchgeo.datasets import RasterDataset, Sentinel2, stack_samples
import torchgeo.models
from torchgeo.samplers import RandomGeoSampler, GridGeoSampler
from torchgeo.samplers.constants import Units

# import matplotlib.pyplot as plt
# import sklearn
# import sklearn.metrics

/n/home10/erolf/.conda/envs/siml_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# todo: wrap train as pytorch lightning tasks


In [21]:
data_dir = "data"

In [14]:
sentinel_data_dir = data_dir+"/sentinel/S2A_MSIL2A_20230418T073611_R092_T36KVU_20230419T022704"

sentinel = Sentinel2(
    sentinel_data_dir,
    bands=["B02", # blue
           "B03", # green
           "B04", # red
           "B08", # NIR
           "TCI"
          ] 
)

In [15]:
nir_band = 3
label_band = 7
vis_band_start = 4
vis_band_end = 7

def my_transforms(sample):
    
    sample['image'][0] = (sample['image'][0] - 1400.) / 60.
    sample['image'][1] = (sample['image'][1] - 1600.) / 60.
    sample['image'][2] = (sample['image'][2] - 1600.) / 120.
    
    sample['image'][nir_band] = (sample['image'][nir_band] - 3000.) / 300.
    return sample

In [16]:
train_site_id = '6'
tch = RasterDataset(root=data_dir+f'/lidar/Karingani_merged_crs_10/site_{train_site_id}')
ds = torchgeo.datasets.IntersectionDataset(sentinel, tch, transforms=my_transforms)


Converting RasterDataset resolution from 9.995824634655532 to 10


In [17]:
# bs = 8
# sampler = RandomGeoSampler(ds, size=64, length=1, units=Units.PIXELS)
# train_dataloader = DataLoader(ds, batch_size=bs, sampler=sampler, collate_fn=stack_samples)

# for sample in train_dataloader:
#     data = sample["image"]

# for img in data:
    
#     fig, ax = plt.subplots(1,3, figsize=(12,4))
#    # ax = [ax]
#     l = ax[0].imshow(img[label_band].numpy(),vmin=0)
#     img_vis = img[vis_band_start:vis_band_end].numpy().astype(int).transpose(1,2,0) 
#     ax[1].imshow(img_vis)
#     nir = ax[2].imshow(img[nir_band].numpy(),vmin=0)
#   #  plt.colorbar(l)
#   #  plt.colorbar(nir)
    
#     ax[0].set_title('Tree height')
#     ax[1].set_title('VIS image')
#     ax[2].set_title('NIR band')
    
# print(img[nir_band].min(), img[nir_band].max())
# print(img[label_band].min(), img[label_band].max())

In [18]:
print(torch.cuda.is_available())

True


In [19]:
device = 'cuda:0'

data_size = 2000
bs = 16
sampler = RandomGeoSampler(ds, size=64, length=data_size, units=Units.PIXELS)
train_dataloader = DataLoader(ds, batch_size=bs, sampler=sampler, collate_fn=stack_samples)

criterion = nn.MSELoss()

model_type = 'fcn'
img_type = 'rgbnir'

if img_type == 'vis':
    in_channels = 3
    img_band_start = vis_band_start
    img_band_end = vis_band_end
elif img_type == 'nir':
    in_channels = 1
    img_band_start = nir_band
    img_band_end = nir_band+1
elif img_type == 'rgb':
    in_channels = 3
    img_band_end = vis_band_end+1
elif img_type == 'rgbnir':
    in_channels = 4
    img_band_start = nir_band
    img_band_end = vis_band_end
    
if model_type == 'fcn_tiny':
    fcn = FCN_tiny(in_channels=in_channels,classes=1)
    pad = 2
elif model_type == 'fcn':    
    fcn = torchgeo.models.FCN(in_channels=in_channels,classes=1).to(device)
    pad = 5


optimizer = optim.SGD(fcn.parameters(), lr=1e-3,  weight_decay=1e-4)

for epoch in range(10):
    running_loss = 0.0
    for i, sample in enumerate(train_dataloader):
        data = sample["image"]
        
      #  img = data[:,nir_band-3:nir_band+1] / 2000.
        img = (data[:,img_band_start:img_band_end]).to(device)
        labels= data[:,label_band:label_band+1,pad:-pad,pad:-pad].to(device)
       # with torch.no_grad():
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = fcn(img)[:,:,pad:-pad,pad:-pad]
        mask = labels > 0
        loss = criterion(outputs[mask], labels[mask])
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
    print(f'{epoch}: {running_loss / (i+1):.2f}')

0: 0.72
1: 0.65
2: 0.64
3: 0.63
4: 0.61
5: 0.62
6: 0.59
7: 0.61
8: 0.58
9: 0.60


In [ ]:
start_epoch = epoch+1    
for epoch in range(start_epoch, start_epoch+100):
    running_loss = 0.0
    for i, sample in enumerate(train_dataloader):
        data = sample["image"]
        
      #  img = data[:,nir_band-3:nir_band+1] / 2000.
        img = data[:,img_band_start:img_band_end].to(device)
        labels= data[:,label_band:label_band+1,pad:-pad,pad:-pad].to(device)
       # with torch.no_grad():
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = fcn(img)[:,:,pad:-pad,pad:-pad]
        mask = labels > 0
        loss = criterion(outputs[mask], labels[mask])
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
    print(f'{epoch}: {running_loss / (i+1):.2f}')

10: 0.57
11: 0.58
12: 0.56
13: 0.56
14: 0.54
15: 0.55
16: 0.56
17: 0.55
18: 0.55
19: 0.54
20: 0.54
21: 0.53
22: 0.54
23: 0.52
24: 0.55
25: 0.54
26: 0.54
27: 0.53
28: 0.52
29: 0.55
30: 0.52
31: 0.53
32: 0.53
33: 0.53
34: 0.53
35: 0.51
36: 0.53
37: 0.51
38: 0.53
39: 0.52
40: 0.51
41: 0.50
42: 0.51
43: 0.52
44: 0.52
45: 0.50
46: 0.51
47: 0.50
48: 0.53
49: 0.51
50: 0.50
51: 0.50
52: 0.50
53: 0.49
54: 0.52
55: 0.53
56: 0.50
57: 0.50
58: 0.49
59: 0.48
60: 0.50
61: 0.49
62: 0.50
63: 0.49
64: 0.49
65: 0.49
66: 0.48
67: 0.49
68: 0.49
69: 0.49
70: 0.50
71: 0.51
72: 0.49
73: 0.49
74: 0.48
75: 0.49
76: 0.49
77: 0.48
78: 0.47
79: 0.48
80: 0.48
81: 0.48
82: 0.47
83: 0.48
84: 0.47
85: 0.47
86: 0.47
87: 0.47
88: 0.48
89: 0.49
90: 0.47
91: 0.46
92: 0.47
93: 0.49
94: 0.48
95: 0.48
96: 0.47
97: 0.47
98: 0.46
99: 0.47
100: 0.46
101: 0.47
102: 0.49
103: 0.47
104: 0.47
